In [1]:
import sys, os
from dotenv import load_dotenv
load_dotenv()
print(sys.executable)
print("Key loaded:", os.getenv("GOOGLE_API_KEY")[:8] + "..." if os.getenv("GOOGLE_API_KEY") else "NOT FOUND")

C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe
Key loaded: AQ.Ab8RN...


In [3]:
import sys
!{sys.executable} -m pip install pypdf numpy langchain-text-splitters google-genai python-dotenv

  Using cached pypdf-6.19.0-py3-none-any.whl.metadata (7.5 kB)
Using cached pypdf-6.19.0-py3-none-any.whl (395 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [5]:
from pypdf import PdfReader

def extract_text_by_page(pdf_path: str) -> list[dict]:
    
    """Returns a list of {page_number, text} dicts."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append({"page_number": i, "text": text.strip()})
    return pages

pages = extract_text_by_page("data/sample_docs/1706.03762v7.pdf")
print(f"Extracted {len(pages)} pages")
print(pages[0]["text"][:500])


Extracted 15 pages
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [6]:
for p in pages:
    print(f"Page {p['page_number']}: {len(p['text'])} characters")

Page 1: 2857 characters
Page 2: 4251 characters
Page 3: 1823 characters
Page 4: 2491 characters
Page 5: 3181 characters
Page 6: 3450 characters
Page 7: 3300 characters
Page 8: 3178 characters
Page 9: 2973 characters
Page 10: 3111 characters
Page 11: 3216 characters
Page 12: 3220 characters
Page 13: 812 characters
Page 14: 815 characters
Page 15: 818 characters


In [7]:

pip install langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_pages(pages: list[dict], chunk_size: int = 1000, chunk_overlap: int = 150) -> list[dict]:
    """Splits each page's text into overlapping chunks, keeping page_number as metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for page in pages:
        page_chunks = splitter.split_text(page["text"])
        for i, chunk_text in enumerate(page_chunks):
            chunks.append({
                "chunk_id": f"page{page['page_number']}_chunk{i}",
                "page_number": page["page_number"],
                "text": chunk_text,
            })
    return chunks

chunks = chunk_pages(pages)
print(f"Total chunks created: {len(chunks)}")
print(chunks[0])

Total chunks created: 49
{'chunk_id': 'page1_chunk0', 'page_number': 1, 'text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, th

In [9]:
print(chunks[5])
print("---")
print(chunks[20])

{'chunk_id': 'page2_chunk1', 'page_number': 2, 'text': 'sequence lengths, as memory constraints limit batching across examples. Recent work has achieved\nsignificant improvements in computational efficiency through factorization tricks [21] and conditional\ncomputation [32], while also improving model performance in case of the latter. The fundamental\nconstraint of sequential computation, however, remains.\nAttention mechanisms have become an integral part of compelling sequence modeling and transduc-\ntion models in various tasks, allowing modeling of dependencies without regard to their distance in\nthe input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms\nare used in conjunction with a recurrent network.\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead\nrelying entirely on an attention mechanism to draw global dependencies between input and output.\nThe Transformer allows for significantly 

In [10]:
smaller_chunks = chunk_pages(pages, chunk_size=500, chunk_overlap=100)
print(f"Total chunks with size=500: {len(smaller_chunks)}")
print(smaller_chunks[5])

Total chunks with size=500: 103
{'chunk_id': 'page1_chunk5', 'page_number': 1, 'text': 'has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head\nattention and the parameter-free position representation and became the other person involved in nearly every\ndetail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and\ntensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and'}


In [11]:
chunks = chunk_pages(pages, chunk_size=1000, chunk_overlap=150)
print(f"Using {len(chunks)} chunks for the rest of the pipeline")

Using 49 chunks for the rest of the pipeline


In [13]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: C:\Users\Vihu\docmind-ai
Files here: ['.env', '.ipynb_checkpoints', '01_pdf_extraction.ipynb', '01_pdf_extraction.py', 'data', 'notebooks', 'Untitled.ipynb', 'untitled.py', 'venv']


In [14]:
import os
print(os.listdir("."))

['.env', '.ipynb_checkpoints', '01_pdf_extraction.ipynb', '01_pdf_extraction.py', 'data', 'notebooks', 'Untitled.ipynb', 'untitled.py', 'venv']


In [15]:
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("GOOGLE_API_KEY")
print("Key loaded:", key[:8] + "..." if key else "NOT FOUND")

Key loaded: AQ.Ab8RN...


In [16]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents="This is a test sentence."
)
print("Embedding length:", len(result.embeddings[0].values))
print("First 5 values:", result.embeddings[0].values[:5])


Embedding length: 3072
First 5 values: [-0.038055073, -0.0028300586, 0.008414354, -0.0674523, -0.010631328]


In [17]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["The cat sat on the mat.", "A feline rested on the rug.", "Stock prices fell sharply today."]
)

emb1 = result.embeddings[0].values
emb2 = result.embeddings[1].values
emb3 = result.embeddings[2].values

print("Similar sentences similarity:", cosine_similarity(emb1, emb2))
print("Unrelated sentences similarity:", cosine_similarity(emb1, emb3))


Similar sentences similarity: 0.7705667422467881
Unrelated sentences similarity: 0.5598135856361813


In [18]:
import sys
print(sys.executable)


C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe


In [19]:
from google import genai
import os
import numpy as np
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["The cat sat on the mat.", "A feline rested on the rug.", "Stock prices fell sharply today."]
)

emb1 = result.embeddings[0].values
emb2 = result.embeddings[1].values
emb3 = result.embeddings[2].values

print("Similar sentences similarity:", cosine_similarity(emb1, emb2))
print("Unrelated sentences similarity:", cosine_similarity(emb1, emb3))

Similar sentences similarity: 0.7705667422467881
Unrelated sentences similarity: 0.5598135856361813


In [20]:
print("Pages:", len(pages))
print("Chunks:", len(chunks))


Pages: 15
Chunks: 49


In [21]:
import sys
!{sys.executable} -m pip install faiss-cpu

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.3 MB 8.5 MB/s eta 0:00:02
   ----- ---------------------------------- 2.4/16.3 MB 7.1 MB/s eta 0:00:02
   ----------- ---------------------------- 4.7/16.3 MB 8.4 MB/s eta 0:00:02
   -------------- ------------------------- 6.0/16.3 MB 8.4 MB/s eta 0:00:02
   ------------------------- -------------- 10.2/16.3 MB 10.8 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.3 MB 10.7 MB/s eta 0:00:01
   ------------------------------------ --- 14.7/16.3 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 10.9 MB/s  0:00:01



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [22]:
def embed_chunks(chunks: list[dict], batch_size: int = 20) -> list[dict]:
    """Embeds each chunk's text, adding an 'embedding' key to each chunk dict."""
    texts = [c["text"] for c in chunks]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=batch
        )
        for j, emb in enumerate(result.embeddings):
            chunks[i + j]["embedding"] = emb.values
    return chunks

chunks = embed_chunks(chunks)
print(f"Embedded {len(chunks)} chunks")
print("Embedding dimension:", len(chunks[0]["embedding"]))

Embedded 49 chunks
Embedding dimension: 3072


In [23]:
import faiss
import numpy as np

def build_faiss_index(chunks: list[dict]):
    embeddings = np.array([c["embedding"] for c in chunks]).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

index = build_faiss_index(chunks)
print(f"FAISS index built with {index.ntotal} vectors")

FAISS index built with 49 vectors


In [24]:
import sys
!{sys.executable} -m pip install faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [25]:
def search(query: str, k: int = 3):
    query_embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query
    ).embeddings[0].values
    query_vector = np.array([query_embedding]).astype("float32")
    distances, indices = index.search(query_vector, k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "chunk": chunks[idx],
            "distance": dist
        })
    return results

results = search("What is multi-head attention?")
for r in results:
    print(f"Page {r['chunk']['page_number']} (distance: {r['distance']:.3f})")
    print(r['chunk']['text'][:200])
    print("---")

Page 5 (distance: 0.438)
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from diff
---
Page 4 (distance: 0.459)
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, 
---
Page 4 (distance: 0.505)
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Multi-Head Attention
Instead of performing a single attention function withdmodel-dimensional keys, va
---
